In [1]:
# imports
import pandas as pd
from pathlib import Path
import requests
import zipfile
import os
os.environ["JAVA_TOOL_OPTIONS"] = "-Xmx8G"
import r5py
import geopandas as gpd
from datetime import datetime, timedelta
from shapely.geometry import Point
import json
import shutil
import numpy as np
from shapely.geometry import box
from datetime import datetime, timedelta

Picked up JAVA_TOOL_OPTIONS: -Xmx8G


In [2]:
# Configuration of paths
URL = "https://data.opentransportdata.swiss/fr/dataset/timetable-2026-gtfs2020/permalink"
GTFS_DIR = Path("data/gtfs")
ZIP_FILE = GTFS_DIR / "gtfs_fp2026.zip"
FILTERED_ZIP_FILE = Path(GTFS_DIR) / "gtfs_filtered_no_taxi.zip"

# Create the directory if it doesn't exist
GTFS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Download the ZIP file if it's not already present
if not ZIP_FILE.exists():
    print(f"Downloading GTFS data to {ZIP_FILE}...")
    # Stream the download to handle large files efficiently
    r = requests.get(URL, stream=True)
    r.raise_for_status() 
    with open(ZIP_FILE, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [4]:
# Extracting all files directly into data/gtfs/
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(GTFS_DIR)
    print(f"Files extracted to {GTFS_DIR}")

Files extracted to data/gtfs


In [5]:
# Configuration for OSM data
OSM_URL = "https://download.geofabrik.de/europe/switzerland-latest.osm.pbf"
OSM_FILE = Path("data/switzerland-latest.osm.pbf")

# Download the PBF file if it's not already present
if not OSM_FILE.exists():
    print(f"Downloading OSM data to {OSM_FILE} (this may take a while)...")
    # Stream the download
    r = requests.get(OSM_URL, stream=True)
    r.raise_for_status()
    
    # On ajoute une petite barre de progression textuelle simple
    total_size = int(r.headers.get('content-length', 0))
    downloaded = 0
    
    with open(OSM_FILE, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024): # 1MB chunks
            f.write(chunk)
            downloaded += len(chunk)
            print(f"\rProgress: {downloaded/(1024*1024):.1f}/{total_size/(1024*1024):.1f} MB", end="")
    print(f"\n✅ OSM data downloaded to {OSM_FILE}")
else:
    print(f"✅ OSM file already exists at {OSM_FILE}")

✅ OSM file already exists at data/switzerland-latest.osm.pbf


In [6]:
SWISS_CRS = "EPSG:2056"   # LV95, meters
WGS84_CRS = "EPSG:4326"   # lon/lat
GRID_SIZE_M = 4000        # 4000 m cells
MAX_TIME_HOURS = 3
DEPARTURE_TIMES = [7, 9, 12, 15, 17, 20]
RUN_DATE = datetime(2026, 3, 20)

In [7]:
cantons_or_communes = gpd.read_file(
    "data/Boundaries_K4_Commune_20260101.gpkg",
    layer="boundaries"
).to_crs("EPSG:2056")

switzerland = cantons_or_communes.dissolve()
switzerland = switzerland.reset_index(drop=True)

switzerland

,geometry,GDENAME,GDEHISTID,GDENR,KTNAME,KTNR,KTKZ
0,"MULTIPOLYGON Z (((2486063 1110424 NaN, 2486306...",Vionnaz,10078,6158,Valais / Wallis,23,VS


In [8]:
switzerland.to_file("data/switzerland_boundary.geojson", driver="GeoJSON")

/Users/ursulael-khoury/miniconda3/envs/dataviz/lib/python3.11/site-packages/pyogrio/raw.py:733: RuntimeWarning: Infinite or NaN coordinate encountered
  ogr_write(


In [9]:
swiss_boundary = switzerland

In [10]:
# Loading the DataFrames
stops = pd.read_csv(GTFS_DIR / "stops.txt")
routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
transfers = pd.read_csv(GTFS_DIR / "transfers.txt", dtype=str)

# Remove Taxi from the type of route and paths
routes = routes[routes["route_type"] != 1500].copy()
trips = trips[trips["route_id"].isin(routes["route_id"])].copy()
stop_times = stop_times[stop_times["trip_id"].isin(trips["trip_id"])].copy()
stops = stops[stops["stop_id"].isin(stop_times["stop_id"])].copy()

valid_stop_ids = set(stops["stop_id"].astype(str))
transfers = transfers[
    transfers["from_stop_id"].isin(valid_stop_ids) &
    transfers["to_stop_id"].isin(valid_stop_ids)
].copy()

print("DataFrames loaded successfully.")

/var/folders/vh/qxbx1gcd7xb1gcxc5z2l5nmw0000gn/T/ipykernel_8845/1933052911.py:2: DtypeWarning: Columns (0: parent_station, 1: platform_code) have mixed types. Specify dtype option on import or set low_memory=False.
  stops = pd.read_csv(GTFS_DIR / "stops.txt")


DataFrames loaded successfully.


In [11]:
# Save new file for network creation
with zipfile.ZipFile(FILTERED_ZIP_FILE, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("routes.txt", routes.to_csv(index=False))
    zf.writestr("trips.txt", trips.to_csv(index=False))
    zf.writestr("stop_times.txt", stop_times.to_csv(index=False))
    zf.writestr("stops.txt", stops.to_csv(index=False))
    zf.writestr("transfers.txt", transfers.to_csv(index=False))

    for fname in [
        "agency.txt", "calendar.txt", "calendar_dates.txt",
        "feed_info.txt", "pathways.txt", "levels.txt",
        "shapes.txt", "frequencies.txt", "attributions.txt"
    ]:
        fpath = Path(GTFS_DIR) / fname
        if fpath.exists():
            zf.write(fpath, arcname=fname)

In [12]:
def build_swiss_grid(boundary_gdf: gpd.GeoDataFrame, cell_size: int = 500):
    xmin, ymin, xmax, ymax = boundary_gdf.total_bounds

    xs = np.arange(xmin, xmax, cell_size)
    ys = np.arange(ymin, ymax, cell_size)

    cells = []
    ids = []

    k = 0
    for x in xs:
        for y in ys:
            cells.append(box(x, y, x + cell_size, y + cell_size))
            ids.append(str(k))
            k += 1

    grid = gpd.GeoDataFrame(
        {"id": ids},
        geometry=cells,
        crs=boundary_gdf.crs
    )

    # Keep only cells intersecting Switzerland
    grid = gpd.overlay(grid, boundary_gdf, how="intersection")

    # Keep a routing point per cell
    origins = grid.copy()
    origins["geometry"] = origins.geometry.centroid

    return grid.reset_index(drop=True), origins.reset_index(drop=True)

In [13]:
grid_cells_4k_lv95, ma_grille_de_points_lv95 = build_swiss_grid(
    swiss_boundary,
    cell_size=GRID_SIZE_M
)

grid_cells_4k_lv95["geometry"] = grid_cells_4k_lv95.geometry.make_valid()
grid_cells_4k_lv95 = grid_cells_4k_lv95[
    grid_cells_4k_lv95.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()
grid_cells_4k_lv95 = grid_cells_4k_lv95[
    ~grid_cells_4k_lv95.geometry.is_empty
].copy()

# only routing points need WGS84
ma_grille_de_points = ma_grille_de_points_lv95.to_crs(WGS84_CRS)

print("Number of 500m cells:", len(grid_cells_4k_lv95))
print("Number of routing origins:", len(ma_grille_de_points))

Number of 500m cells: 2813
Number of routing origins: 2813


In [15]:
OUTPUTS_DIR = Path("outputs/outputs_nearest_ic")

In [16]:
grid_cells_4k_lv95.to_parquet(
    OUTPUTS_DIR / "grid_cells_4k_lv95.parquet",
    index=False
)

In [17]:
ma_grille_de_points.to_parquet(
    OUTPUTS_DIR / "grid_points_4k.parquet",
    index=False
)

In [18]:
# We look for routes that are InterCity (IC) trains based on their route_desc 'IC'
ic_routes = routes[routes['route_desc'] == 'IC']
ic_trips = trips[trips['route_id'].isin(ic_routes['route_id'])]

# Get the stop_id where IC trains actually stop
ic_stop_ids = stop_times[stop_times['trip_id'].isin(ic_trips['trip_id'])]['stop_id'].unique()

In [19]:
# Keep the IC stop rows first
ic_stops = stops[stops["stop_id"].isin(ic_stop_ids.astype(str))].copy()

ic_stops["station_id"] = ic_stops["parent_station"].where(
    ic_stops["parent_station"].notna() & (ic_stops["parent_station"] != ""),
    ic_stops["stop_id"]
).astype(str)

# Remove the prefix "Parent" when it exists
ic_stops["station_id"] = ic_stops["station_id"].str.replace(r"^Parent", "", regex=True)

# Build station-level destinations from the parent station rows
ic_station_ids = ic_stops["station_id"].dropna().unique()

print("Matches in stops:",
      stops["stop_id"].astype(str).isin(ic_station_ids).sum())

ic_stations = stops[stops["stop_id"].astype(str).isin(ic_station_ids)].copy()
ic_stations = ic_stations.drop_duplicates(subset=["stop_id"]).copy()

mes_gares_ic = gpd.GeoDataFrame(
    {
        "id": ic_stations["stop_id"].astype(str),
        "stop_id": ic_stations["stop_id"].astype(str),
        "stop_name": ic_stations["stop_name"],
    },
    geometry=gpd.points_from_xy(
        ic_stations["stop_lon"].astype(float),
        ic_stations["stop_lat"].astype(float)
    ),
    crs="EPSG:4326"
).reset_index(drop=True)

print(len(mes_gares_ic))
mes_gares_ic.head()

Matches in stops: 134
134


,id,stop_id,stop_name,geometry
0,8014325,8014325,Ringsheim/Europa-Park,POINT (7.77304 48.24872)
1,8014558,8014558,Singen (Hohentwiel),POINT (8.84127 47.75888)
2,8301003,8301003,Domodossola (I),POINT (8.29624 46.11529)
3,8500010,8500010,Basel SBB,POINT (7.58956 47.54741)
4,8500023,8500023,Liestal,POINT (7.73137 47.48446)


In [20]:
mes_gares_ic = mes_gares_ic.copy()
mes_gares_ic["id"] = mes_gares_ic["id"].astype(str)

if mes_gares_ic.crs is None:
    mes_gares_ic = mes_gares_ic.set_crs(WGS84_CRS)
else:
    mes_gares_ic = mes_gares_ic.to_crs(WGS84_CRS)

print("IC destinations:", len(mes_gares_ic))
mes_gares_ic.head()

IC destinations: 134


,id,stop_id,stop_name,geometry
0,8014325,8014325,Ringsheim/Europa-Park,POINT (7.77304 48.24872)
1,8014558,8014558,Singen (Hohentwiel),POINT (8.84127 47.75888)
2,8301003,8301003,Domodossola (I),POINT (8.29624 46.11529)
3,8500010,8500010,Basel SBB,POINT (7.58956 47.54741)
4,8500023,8500023,Liestal,POINT (7.73137 47.48446)


In [21]:
# Test de la présence de Java (indispensable pour r5py)
java_check = os.popen('java -version 2>&1').read()

if "version" in java_check.lower():
    print(f"✅ Succès : Java est bien lié à l'env 'dataviz'.\nDétails : {java_check.splitlines()[0]}")
    print(f"✅ r5py est prêt à l'emploi.")
else:
    print("❌ Erreur : Java n'est pas détecté dans cet environnement.")
    print("Conseil : Tape 'conda install -c conda-forge openjdk' dans ton terminal.")

✅ Succès : Java est bien lié à l'env 'dataviz'.
Détails : Picked up JAVA_TOOL_OPTIONS: -Xmx8G
✅ r5py est prêt à l'emploi.


In [22]:
# Initialiser le réseau (Longue étape, car il compile le graphe)
network = r5py.TransportNetwork(
    str(OSM_FILE),
    [str(FILTERED_ZIP_FILE)]
)

In [27]:
TRAVEL_TIMES_FILE = Path("outputs/outputs_nearest_ic/travel_times.parquet")

if False:#if TRAVEL_TIMES_FILE.exists():
    print("Loading saved travel times...")
    computer = pd.read_parquet(TRAVEL_TIMES_FILE)
else:
    print("Computing travel times...")
    computer = r5py.TravelTimeMatrix(
        network,
        origins=ma_grille_de_points,
        destinations=mes_gares_ic,
        departure=datetime(2026, 3, 20, 8, 0),
        departure_time_window=timedelta(minutes=60),
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
        percentiles=[50],
        snap_to_network=True,
        max_time=timedelta(hours=4),
    )
    computer.to_parquet(TRAVEL_TIMES_FILE, index=False)
    print("Saved to", TRAVEL_TIMES_FILE)

Computing travel times...


/Users/ursulael-khoury/miniconda3/envs/dataviz/lib/python3.11/site-packages/r5py/r5/base_travel_time_matrix.py:231: RuntimeWarning: Some origin points could not be snapped to the street network
  warnings.warn(
/Users/ursulael-khoury/miniconda3/envs/dataviz/lib/python3.11/site-packages/r5py/r5/base_travel_time_matrix.py:231: RuntimeWarning: Some destination points could not be snapped to the street network
  warnings.warn(


Saved to outputs/outputs_nearest_ic/travel_times.parquet


In [28]:
print("Non-NaN:", computer["travel_time"].notna().sum())
print("Total:", len(computer))

print("\nValid destinations per origin:")
print(
    computer.groupby("from_id")["travel_time"]
    .apply(lambda s: s.notna().sum())
    .sort_values()
    .head(10)
)

print("\nValid origins per destination:")
print(
    computer.groupby("to_id")["travel_time"]
    .apply(lambda s: s.notna().sum())
    .sort_values()
    .head(10)
)

Non-NaN: 78376
Total: 192990

Valid destinations per origin:
from_id
2707    0
2811    0
3424    0
1364    0
1363    0
1850    0
1360    0
1359    0
2597    0
1854    0
Name: travel_time, dtype: int64

Valid origins per destination:
to_id
8505307    344
8505305    377
8505400    438
8505209    499
8505300    547
8501026    553
8505404    572
8505214    645
8501008    669
8505213    724
Name: travel_time, dtype: int64


In [30]:
PUBLIC_DATA_DIR = Path("public/data")
REACHABLE_FILE = PUBLIC_DATA_DIR / "reachable_from_all.json"
PROCESSED_DATA_DIR = Path("data/processed")

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
PUBLIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save core inputs
# ma_grille_de_points.to_parquet(OUTPUTS_DIR / "grid_points.parquet", index=False)
ma_grille_de_points.to_parquet("outputs/outputs_nearest_ic/grid_points_100m.parquet", index=False)
mes_gares_ic.to_parquet(OUTPUTS_DIR / "ic_stations.parquet", index=False)

# optional: also save filtered GTFS copy in the repo structure
if "FILTERED_ZIP_FILE" in globals():
    shutil.copy(FILTERED_ZIP_FILE, PROCESSED_DATA_DIR / "gtfs_filtered_no_taxi.zip")

# Save full travel-time matrix
travel_times = computer.copy()
# travel_times.to_parquet(OUTPUTS_DIR / "travel_times_6h.parquet", index=False)
travel_times.to_parquet("outputs/outputs_nearest_ic/travel_times_100m_6h.parquet", index=False)

# Keep only valid OD pairs within 6h
valid_6h = (
    travel_times
    .dropna(subset=["travel_time"])
    .copy()
)

valid_6h["from_id"] = valid_6h["from_id"].astype(str)
valid_6h["to_id"] = valid_6h["to_id"].astype(str)

valid_6h = valid_6h[valid_6h["travel_time"] <= 360].copy()
valid_6h.to_parquet(OUTPUTS_DIR / "od_pairs_6h.parquet", index=False)

# Accessibility summary per origin
summary = (
    valid_6h.groupby("from_id", as_index=False)
    .agg(
        min_travel_time_to_ic=("travel_time", "min"),
        n_destinations_reachable_6h=("to_id", "nunique"),
    )
)

reachable_60 = (
    travel_times
    .dropna(subset=["travel_time"])
    .copy()
)
reachable_60["from_id"] = reachable_60["from_id"].astype(str)
reachable_60["to_id"] = reachable_60["to_id"].astype(str)

reachable_60 = (
    reachable_60[reachable_60["travel_time"] <= 60]
    .groupby("from_id", as_index=False)
    .agg(n_ic_reachable_60=("to_id", "nunique"))
)

summary = summary.merge(reachable_60, on="from_id", how="left")

accessibility_summary = ma_grille_de_points.copy()
accessibility_summary["id"] = accessibility_summary["id"].astype(str)

accessibility_summary = accessibility_summary.merge(
    summary,
    left_on="id",
    right_on="from_id",
    how="left"
)

accessibility_summary["n_destinations_reachable_6h"] = (
    accessibility_summary["n_destinations_reachable_6h"].fillna(0).astype(int)
)
accessibility_summary["n_ic_reachable_60"] = (
    accessibility_summary["n_ic_reachable_60"].fillna(0).astype(int)
)

accessibility_summary.to_parquet(
    OUTPUTS_DIR / "accessibility_summary.parquet",
    index=False
)

# Save overlay for Leaflet
accessibility_summary.to_file(
    PUBLIC_DATA_DIR / "accessibility_overlay.geojson",
    driver="GeoJSON"
)

# optional: also save IC hubs as GeoJSON for the map
mes_gares_ic.to_file(
    PUBLIC_DATA_DIR / "ic_hubs.geojson",
    driver="GeoJSON"
)

# Save one JSON per origin for click interaction
dest_lookup = mes_gares_ic[["id", "stop_name", "geometry"]].copy()
dest_lookup["id"] = dest_lookup["id"].astype(str)
dest_lookup["lon"] = dest_lookup.geometry.x
dest_lookup["lat"] = dest_lookup.geometry.y
dest_lookup = dest_lookup.drop(columns="geometry")

valid_6h_export = valid_6h.merge(
    dest_lookup,
    left_on="to_id",
    right_on="id",
    how="left"
)

reachable_from_all = {}

for origin_id, group in valid_6h_export.groupby("from_id"):
    group = group.sort_values("travel_time").copy()

    reachable_from_all[str(origin_id)] = {
        "origin_id": str(origin_id),
        "destinations": group[
            ["to_id", "stop_name", "travel_time", "lat", "lon"]
        ].to_dict(orient="records")
    }

with open(PUBLIC_DATA_DIR / "reachable_from_all.json", "w", encoding="utf-8") as f:
    json.dump(reachable_from_all, f, ensure_ascii=False)

# Save metadata
metadata = {
    "departure": "2026-03-20 08:00",
    "departure_time_window_minutes": 60,
    "max_time_minutes": 360,
    "transport_modes": ["WALK", "TRANSIT"],
    "percentiles": [50],
    "grid_resolution_degrees": 0.05,
    "osm_file": str(OSM_FILE) if "OSM_FILE" in globals() else None,
    "gtfs_file": "data/processed/gtfs_filtered_no_taxi.zip",
    "n_origins": int(len(ma_grille_de_points)),
    "n_destinations": int(len(mes_gares_ic)),
    "n_rows_matrix": int(len(travel_times)),
    "n_valid_rows_6h": int(len(valid_6h)),
    "n_overlay_points": int(len(accessibility_summary)),
}

with open(OUTPUTS_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved:")
print("-", OUTPUTS_DIR / "grid_points.parquet")
print("-", OUTPUTS_DIR / "ic_stations.parquet")
print("-", OUTPUTS_DIR / "travel_times_6h.parquet")
print("-", OUTPUTS_DIR / "od_pairs_6h.parquet")
print("-", OUTPUTS_DIR / "accessibility_summary.parquet")
print("-", PUBLIC_DATA_DIR / "accessibility_overlay.geojson")
print("-", PUBLIC_DATA_DIR / "ic_hubs.geojson")
print("-", OUTPUTS_DIR / "run_metadata.json")

Saved:
- outputs/outputs_nearest_ic/grid_points.parquet
- outputs/outputs_nearest_ic/ic_stations.parquet
- outputs/outputs_nearest_ic/travel_times_6h.parquet
- outputs/outputs_nearest_ic/od_pairs_6h.parquet
- outputs/outputs_nearest_ic/accessibility_summary.parquet
- public/data/accessibility_overlay.geojson
- public/data/ic_hubs.geojson
- outputs/outputs_nearest_ic/run_metadata.json


In [31]:
# Exécuter
travel_times = computer

travel_times.shape

(192990, 3)

In [32]:
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime, timedelta
import r5py

test_origin = gpd.GeoDataFrame(
    {"id": ["lausanne_test"]},
    geometry=[Point(6.6323, 46.5197)],
    crs="EPSG:4326"
)

test_dest = mes_gares_ic[mes_gares_ic["id"].astype(str) == "8503000"].copy()  # Zürich HB

test_matrix = r5py.TravelTimeMatrix(
    network,
    origins=test_origin,
    destinations=test_dest,
    departure=datetime(2026, 3, 20, 8, 0),
    departure_time_window=timedelta(minutes=60),
    transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    percentiles=[50],
    snap_to_network=True,
    max_time=timedelta(hours=4),
)

test_matrix

,from_id,to_id,travel_time
0,lausanne_test,8503000,166
